Copyright 2026 Google LLC

In [ ]:
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

<a target="_blank" href="https://colab.research.google.com/github/lucianommartins/lab-sabadao/blob/main/examples/notebooks/02_workloads_campaigns_and_concurrency.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# gbench workload campaigns, concurrency scaling, and prompt geometry

**Author:** [Luciano Martins](https://github.com/lucianommartins)

This notebook explores how to evaluate foundation model performance under realistic production workloads using `gbench`. You will test concurrency scaling by sweeping batch sizes, configure custom prompt geometries, and run predefined performance campaigns (`chat-like`, `agentic`, `decode-heavy`) against a local Ollama serving engine.

## Learning objectives

1. Configure Ollama to serve a quantized Google Gemma 4 model (`unsloth/gemma-4-E4B-it-qat-GGUF`) with a context window of 8192 tokens.
2. Execute concurrency sweeps (`--batch-sizes 1 4 8 16`) to measure latency and throughput scaling curves.
3. Test custom input and output prompt token lengths (`--input-lengths` and `--output-lengths`).
4. Execute and compare standardized production workload campaigns (`chat-like`, `agentic`, `decode-heavy`).
5. Terminate the local Ollama session and reclaim system memory.

## Useful resources

* [lab-sabadao GitHub repository](https://github.com/lucianommartins/lab-sabadao)
* [Ollama documentation](https://github.com/ollama/ollama)
* [ShareGPT V3 dataset](https://huggingface.co/datasets/anon8231489123/ShareGPT_Vicuna_unfiltered)

## 1. Environment setup and installation

We clone the `lab-sabadao` repository from GitHub, change directory into the project root (`%cd lab-sabadao`), and install the package in editable mode (`%pip install -e .`). This builds and links the `gbench` CLI executable without installing unnecessary development linters.

In [ ]:
import os
from pathlib import Path

# Safe environment setup (prevents recursive nesting on re-run)
if not Path("pyproject.toml").is_file() and not Path("gbench").is_dir():
    if not Path("lab-sabadao").is_dir():
        !git clone https://github.com/lucianommartins/lab-sabadao.git
    %cd lab-sabadao

%pip install -e . -q
import gbench
print(f"gbench version {gbench.__version__} installed successfully.")

# Inspect available presets
!gbench --list presets

## 2. Installing Ollama locally

We check if the Ollama binary is present on the system. If it is not found, we install Ollama using its official Linux installation script (`curl -fsSL https://ollama.com/install.sh | sh`). Finally, we run `ollama --version` to verify that the installation succeeded and the CLI is available.

In [ ]:
import subprocess, os, shutil

if not shutil.which("ollama"):
    print("Installing Ollama locally...")
    # Ensure zstd is available (required by Ollama Linux tar.zst packages)
    subprocess.run("command -v zstd >/dev/null || (command -v apt-get >/dev/null && apt-get update -qq && apt-get install -y -qq zstd)", shell=True)
    # Run official Ollama installer
    subprocess.run("curl -fsSL https://ollama.com/install.sh | sh", shell=True)
else:
    print("Ollama binary already installed.")

# Ensure binary directory is present in PATH for subsequent cells
for p in ["/usr/local/bin", "/usr/bin", os.path.expanduser("~/.local/bin")]:
    if os.path.exists(os.path.join(p, "ollama")) and p not in os.environ.get("PATH", ""):
        os.environ["PATH"] = f"{p}:{os.environ.get('PATH', '')}"

!ollama --version

## 3. Launching background Ollama server

We launch the `ollama serve` process in the background and send a health check request to `http://localhost:11434/` to verify that the HTTP API is alive ("Ollama is running").

In [ ]:
import subprocess, time, requests
try:
    resp = requests.get("http://localhost:11434/", timeout=2)
    print("Ollama server already active:", resp.text.strip())
except Exception:
    print("Starting background ollama serve...")
    subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    time.sleep(4)
    resp = requests.get("http://localhost:11434/")
    print("Server health check:", resp.text.strip())

## 4. Writing custom Modelfile for QAT model

We create an Ollama `Modelfile.qat` that configures our quantized Google Gemma 4 model (`hf.co/unsloth/gemma-4-E4B-it-qat-GGUF:latest`) with explicit parameters:
* **`num_ctx 8192`**: Context window of 8192 tokens.
* **`SYSTEM prompt`**: System instruction defining Gemma 4 AI assistant capabilities.

In [ ]:
HF_MODEL_ID = "hf.co/unsloth/gemma-4-E4B-it-qat-GGUF:UD-Q4_K_XL"
modelfile_content = f"""FROM {HF_MODEL_ID}
PARAMETER num_ctx 8192
SYSTEM "You are a helpful Gemma 4 AI assistant with reasoning, vision, and tool calling capabilities."
"""
with open("Modelfile.qat", "w", encoding="utf-8") as f:
    f.write(modelfile_content)
print("Created Modelfile.qat with valid Ollama parameters (num_ctx 8192, SYSTEM prompt).")

## 5. Registering model and running generation smoke test

We register our custom model tag (`gemma4-qat:4b`) using `ollama create -f Modelfile.qat`. This pulls the GGUF weights from Hugging Face Hub if not already cached. We then run a quick generation test (`ollama run`) to verify that the model loads into hardware memory and generates tokens correctly.

In [ ]:
MODEL_TAG = "gemma4-qat:4b"
print(f"Registering model {MODEL_TAG} from Modelfile.qat...")
!ollama create {MODEL_TAG} -f Modelfile.qat
print("Running quick generation smoke test...")
!ollama run {MODEL_TAG} "Reply with the single word: READY."

## 6. Verifying OpenAI REST endpoint readiness

Before launching `gbench`, we query `http://localhost:11434/v1/models` to verify that Ollama is serving standard OpenAI `/v1` REST payloads and that our registered model is listed.

In [ ]:
import requests
resp = requests.get("http://localhost:11434/v1/models")
print("OpenAI /v1/models endpoint HTTP status:", resp.status_code)
models = [m["id"] for m in resp.json().get("data", [])]
print("Available REST models:", models)

## 7. Concurrency scaling sweeps

To evaluate how latency and throughput scale under increasing client concurrency, we pass `--batch-sizes 1 4 8 16`. This executes 4 sequential serving benchmark runs against the Ollama endpoint, measuring the Time to First Token (TTFT) degradation and token throughput saturation curve.

In [ ]:
!gbench --models gemma4-qat:4b \
        --batch-sizes 1 4 8 \
        --remote-endpoint http://localhost:11434/v1 \
        --tokenizer google/gemma-4-E4B-it \
        --results-dir ./results_concurrency \
        --num-iterations 1 --warmup-iterations 0 --num-prompts 10

## 8. Standardized production workload campaigns

`gbench` includes 6 predefined workload campaigns that simulate production prompt geometries:
* **`chat-like`**: Multi-turn conversational traffic from the [ShareGPT V3 dataset](https://huggingface.co/datasets/anon8231489123/ShareGPT_Vicuna_unfiltered).
* **`agentic`**: Long-context prompt ingestion (`8000` tokens) followed by a concise tool call response (`400` tokens).
* **`decode-heavy`**: Short prompt processing (`128` tokens) followed by high-volume generation (`2048` tokens).
* **`prefill-heavy`**: Long prompt ingestion (`8192` tokens) followed by short generation (`128` tokens).
* **`mixed`**: Balanced intermediate prompt processing (`4096` tokens / `1024` tokens).
* **`long-decode`**: High-context long-form generation (`8192` tokens / `8192` tokens).

We can inspect all built-in benchmark pillars via `!gbench --list pillars` or presets via `!gbench --list presets`, and then execute `agentic` and `decode-heavy` campaigns against our Ollama QAT model.

In [ ]:
# List available benchmark presets
!gbench --list presets

# Execute agentic and decode-heavy production campaigns
!gbench --models gemma4-qat:4b \
        --campaign agentic \
        --remote-endpoint http://localhost:11434/v1 \
        --tokenizer google/gemma-4-E4B-it \
        --results-dir ./results_campaigns \
        --num-prompts 5 --num-iterations 1

!gbench --models gemma4-qat:4b \
        --campaign decode-heavy \
        --remote-endpoint http://localhost:11434/v1 \
        --tokenizer google/gemma-4-E4B-it \
        --results-dir ./results_campaigns \
        --num-prompts 5 --num-iterations 1

## 9. Visualizing concurrency scaling curves

We use `matplotlib` to plot how Time to First Token (TTFT) and Time per Output Token (TPOT) increase as client concurrency (`batch_size`) scales from 1 to 16.

In [ ]:
import json, glob, os
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

results_base = Path("./results_concurrency")
run_dirs = sorted([d for d in results_base.iterdir() if d.is_dir()], key=lambda d: d.stat().st_mtime, reverse=True) if results_base.exists() else []

if run_dirs:
    latest_dir = run_dirs[0]
    summary_path = latest_dir / "summary.json"
    
    if summary_path.exists():
        with open(summary_path, "r", encoding="utf-8") as f:
            data = json.load(f)
        df = pd.DataFrame(data.get("models", []))
    else:
        # Fallback: load individual performance run JSON files
        perf_files = sorted(latest_dir.glob("performance/serve_*.json"))
        records = []
        for pf in perf_files:
            with open(pf, "r", encoding="utf-8") as f:
                records.append(json.load(f))
        df = pd.DataFrame(records)

    if not df.empty and "batch_size" in df.columns:
        df = df.sort_values("batch_size")
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
        
        ttft_col = "median_ttft_ms" if "median_ttft_ms" in df.columns else ("mean_ttft_ms" if "mean_ttft_ms" in df.columns else "ttft_p50_ms")
        ax1.plot(df["batch_size"], df[ttft_col], marker="o", color="blue")
        ax1.set_title("TTFT Latency vs Concurrency")
        ax1.set_xlabel("Concurrent Batch Size")
        ax1.set_ylabel("TTFT (ms)")
        ax1.grid(True)
        
        tp_col = "output_token_throughput" if "output_token_throughput" in df.columns else "request_throughput"
        ax2.plot(df["batch_size"], df[tp_col], marker="s", color="green")
        ax2.set_title("Total Output Throughput vs Concurrency")
        ax2.set_xlabel("Concurrent Batch Size")
        ax2.set_ylabel("Output Tokens / sec")
        ax2.grid(True)
        
        plt.tight_layout()
        plt.show()
    else:
        print("No concurrency sweep records found in:", latest_dir)
else:
    print("No concurrency sweep directory found.")

## 10. Session cleanup and server shutdown

We terminate the background Ollama process and delete temporary test Modelfiles.

In [ ]:
import subprocess, os

subprocess.run(["pkill", "-f", "ollama"], check=False)
if os.path.exists("Modelfile.qat"):
    os.remove("Modelfile.qat")
print("Session cleanup complete.")